# 51 · The MERS-CoV example's epidemiology, checked against pandas

## What this checks

This check covers the epidemiology in a whole worked example, not one widget. The MERS-CoV example, on the 2015
outbreak in South Korea, has three workflows that compute it:

- **Deaths by age and sex.** Deaths by age class, and by sex in a two-by-two table.
- **Days from exposure.** The days from exposure to onset of illness, and from onset to death. Each is worked out by
  a rule, with the median for each outcome.
- **Secondary cases.** The cases each source infected, and the median days between onsets, by setting.

These workflows chain these widgets:

- **Total within groups**, with its Mean, Median and a column to total;
- **Apply ruleset**, for the days between dates;
- **Two-by-two**.

The result is recorded against Total within groups release 0.7.0. Its Mean, Median and column to total were new
at release 0.4.0, when this check first tested them independently.

## How it is checked

We recompute every value from the example's two source files, using **pandas**, Python's library for tables. pandas
counts the days between calendar dates, counts the cases and takes the medians itself. A median is the middle value
once the values are in order.

The two-by-two's estimates come from the textbook formulas:

- **Risk ratio.** How many times more likely the outcome is in one group than in the other. Its interval is Katz's,
  worked out on the log scale.
- **Odds ratio.** The same comparison made with odds, not risks. Its interval is Woolf's, also on the log scale.
- **Risk difference.** How much higher the risk is in one group, as a difference. Its interval is the unpooled Wald
  interval.

Each interval is a 95% confidence interval: the range of values the data are consistent with. The normal-curve value
these formulas need comes from Python's own standard library. Nothing here reads Fieldwork's tables.

**The data are not in this lab**, as in check 49. The source, the dataset `mers_korea_2015`, states no licence. The lab keeps Fieldwork's results,
the commit the example pins, and a digest of each file. A digest is a fingerprint of a file's exact contents. The
files are read at that commit, and each digest is checked first.

## What should happen

1. **Each case's days**, from exposure to onset and from onset to death. Exact, and empty where a date is empty.
2. **Every count.** Exact.
3. **Every share.** Exact, under the rounding Fieldwork declares: a percentage to one decimal place, with a half
   rounded up.
4. **Every median.** Exact. It is the middle value of those present, or the mean of the middle two. A group with an
   empty value, not left out, has no median.
5. **The two-by-two.** The four cells exact. Each estimate and both limits of its interval within one part in a
   billion (10⁻⁹).

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the installed packages.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_51_mers_epi import run

report = run(Path.cwd() / "fixtures" / "mers-epi-001.json")
print("agrees:", report["agrees"])
print("checked:", report["checked"])
print("source:", report["source"]["commit"][:7])
print("differs:", report["differs"])
print("observed:", report["observed"])

## The result on record

First recorded on 14 November 2026, against Total within groups release 0.4.0. Rerun on 26 November 2026, against
Total within groups release 0.7.0: **Agrees**.

The three workflows were recomputed from the files at the example's commit bc2d8f3. Every value agreed:

- 324 days, from exposure to onset and from onset to death;
- 50 counts and 24 shares;
- 7 medians, two of them of a column worked out by a rule;
- the two-by-two's 9 estimates and limits.

The check was also shown able to fail. In a copy of the data, a median, a share and an interval limit were changed.
The check found each one.

Running the cell above repeats the check on the same data.

## What this does not show

- That the days from the last day of exposure to onset are an incubation period, the time from infection to the
  first symptoms. 3 cases have an onset before their exposure window closed. Both sides keep them as written, so
  some days are negative.
- That `diff_dt_onset`, the column of days between onsets, is a serial interval, the time from one case's onset to the onset of the
  case they infected. The source does not document how the links or their days were set.
- That the risk ratio by sex says anything about sex. Its interval holds 1, and no study design is declared.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
e = report["environment"]
print("pandas:", e["pandas"], "·", e["runtime"], "· Pyodide" if e["pyodide"] else "")